# 10 — Schema linking на 60 таблицах

> **Инженерный вызов:** генератор не «видит» схему → галлюцинирует имена таблиц → EX падает до 30-40%.
>
> **Цель:** дать в промпт **только релевантные** таблицы (top-15 + FK-замыкание), не всю схему сразу.

## Что мы покажем

1. Соберём мок-каталог из 60 таблиц (имитация `data_model_sql/data_model.sql`).
2. Реализуем **bag-of-words эмбеддинг** через `Counter` (без внешних зависимостей).
3. Найдём топ-15 таблиц по NL-вопросу через cosine similarity.
4. Сделаем **замыкание по FK** — добавим справочники.
5. Сравним: «вся схема в промпт» vs «только релевантные».


## 🧒 Аналогия для ребёнка

Ты пришёл в **огромную библиотеку** со 60 полками книг. Нужна
книга про динозавров.

- **Плохо:** взять **все 60 полок целиком** и тащить в комнату
  для чтения. Стол не вмещает, нужное теряется среди ненужного.
- **Хорошо:** идёшь к **картотеке**, говоришь «динозавры», получаешь
  15 номеров полок. Идёшь именно туда. Полка про «эпоху мезозоя»
  ссылается на полку «карта мира того времени» — берёшь и её
  тоже (это **FK-замыкание**).

Schema linking = картотека для БД.


## 1. Setup — мок-каталог из 60 таблиц


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Мок-каталог из 60 таблиц банк/ERP домена.
# @details
#   В реальной системе мы читаем data_model_sql/data_model.sql и парсим
#   CREATE TABLE + COMMENT ON. Здесь — упрощённо: список туплов.
#   Поля: (имя, русский комментарий, [колонки], [FK к другим таблицам])
SCHEMA_CATALOG = [
    ("clients",          "Клиенты банка",                 ["client_id", "full_name", "passport", "phone", "balance"], []),
    ("acc_number",       "Номер счета клиента",           ["id", "client_id", "account_name", "type_id"], ["clients", "acc_type"]),
    ("acc_type",         "Тип счета (справочник)",        ["id", "name", "description"], []),
    ("credit_contract",  "Кредитный договор",             ["id", "client_id", "amount", "rate", "status_id"], ["clients", "credit_status"]),
    ("credit_status",    "Статус кредита (справочник)",   ["id", "name"], []),
    ("payment",          "Платёж по кредиту",             ["id", "contract_id", "amount", "date"], ["credit_contract"]),
    ("business_segment", "Бизнес-сегмент клиента",        ["id", "client_id", "segment_name"], ["clients"]),
    ("ic_application",   "Заявление на ипотеку",          ["id", "client_id", "amount", "created_at"], ["clients"]),
    ("mler_application", "Заявление на MLEr",             ["id", "client_id", "status"], ["clients"]),
    ("offices_psb",      "Офисы ПСБ",                     ["id", "name", "address", "city"], []),
    ("employees",        "Сотрудники банка",              ["id", "full_name", "office_id"], ["offices_psb"]),
    ("dict_product",     "Справочник банковских продуктов", ["id", "name", "type"], []),
    ("transaction_log",  "Журнал транзакций",             ["id", "account_id", "amount", "ts"], ["acc_number"]),
    ("cb_interest_rate", "Ставки ЦБ",                     ["id", "rate", "valid_from"], []),
    ("fs_file",          "Файлы-приложения",              ["id", "object_id", "path", "size"], []),
    ("count_turnover",   "Оборот по счёту",               ["id", "account_id", "month", "amount"], ["acc_number"]),
    ("participant_app",  "Заявление участника",           ["id", "client_id", "type"], ["clients"]),
    ("user_log",         "Журнал действий пользователей", ["id", "user_id", "action", "ts"], []),
    ("audit_event",      "Аудит-событие",                 ["id", "action", "user_id", "ts"], []),
    ("kpi_report",       "KPI-отчёт сотрудника",          ["id", "employee_id", "month", "value"], ["employees"]),
]
# Добавим ещё 40 «технических» таблиц вроде ms_<hash> с минимальным контекстом
for i in range(40):
    SCHEMA_CATALOG.append((f"ms_table_{i:03d}", f"Системная таблица №{i}", ["id", "name", "value"], []))

print(f"Всего таблиц в каталоге: {len(SCHEMA_CATALOG)}")
print(f"Первые 5:")
for t in SCHEMA_CATALOG[:5]:
    print(f"  {t[0]:20s}  {t[1]}")


## 2. Bag-of-words эмбеддинг

Реальная система использует `intfloat/multilingual-e5-large` —
нейросеть, которая выдаёт 1024-мерный вектор по тексту.

Здесь — **упрощённая версия** через `Counter`: считаем сколько
раз каждое слово встречается в «описании» таблицы. Это
достаточно для демо принципа.


In [ ]:
from collections import Counter
import math


##
# @brief Превращает текст в bag-of-words вектор.
# @param text  Любой текст (имя таблицы + комментарий + колонки).
# @return      Counter: слово → частота.
# @note  В проде вместо этого — sentence-transformers и FAISS.
def embed(text):
    # Приводим к нижнему регистру, разбиваем по не-буквам, убираем короткие
    words = re.findall(r"[a-zA-Zа-яА-Я]{3,}", text.lower())
    return Counter(words)


def table_text(table_tuple):
    name, comment, cols, fks = table_tuple
    return f"{name} {comment} {' '.join(cols)}"


##
# @brief Cosine similarity между двумя bag-of-words.
def cosine(a, b):
    common = set(a) & set(b)
    if not common:
        return 0.0
    num = sum(a[w] * b[w] for w in common)
    norm_a = math.sqrt(sum(v * v for v in a.values()))
    norm_b = math.sqrt(sum(v * v for v in b.values()))
    return num / (norm_a * norm_b) if norm_a and norm_b else 0.0


section("Эмбеддим все 60 таблиц")
table_vectors = [(t[0], embed(table_text(t))) for t in SCHEMA_CATALOG]
print(f"Вектор таблицы 'clients': {dict(list(table_vectors[0][1].items())[:5])} ... ({len(table_vectors[0][1])} слов)")


## 3. Retrieval: top-15 по NL-вопросу

Пользователь спрашивает по-русски, система ищет релевантные
таблицы. Реальный пример из домена банка.


In [ ]:
##
# @brief Возвращает top-k таблиц по релевантности к вопросу.
def schema_link(question, k=15):
    q_vec = embed(question)
    scored = [(name, cosine(q_vec, vec)) for name, vec in table_vectors]
    scored.sort(key=lambda x: -x[1])
    return scored[:k]


section("Запрос: 'покажи мне всех клиентов с просроченными кредитами'")
top = schema_link("покажи мне всех клиентов с просроченными кредитами банка")
for name, score in top:
    print(f"  {score:.3f}  {name}")


## 4. FK-замыкание

Top-15 может содержать `credit_contract`, но **забыть** `clients`,
потому что само слово «клиент» в каталоге `credit_contract` слабо
представлено. FK-замыкание решает: если таблица A ссылается на B,
обе нужны для JOIN.


In [ ]:
##
# @brief Расширяет топ таблицами, на которые они ссылаются по FK.
def fk_closure(selected_names):
    catalog_by_name = {t[0]: t for t in SCHEMA_CATALOG}
    result = set(selected_names)
    for name in list(result):
        if name not in catalog_by_name:
            continue
        for fk_target in catalog_by_name[name][3]:
            result.add(fk_target)
    return result


section("FK-замыкание для top-3")
top3 = [n for n, _ in top[:3]]
print(f"Top-3:                {top3}")
closed = fk_closure(top3)
print(f"После FK-замыкания:   {sorted(closed)}")


## 5. Сравнение бюджета токенов


In [ ]:
def schema_to_prompt(table_names):
    """Превращает список таблиц в DDL-фрагмент (грубая оценка токенов = символы/4)."""
    catalog_by_name = {t[0]: t for t in SCHEMA_CATALOG}
    out = []
    for name in table_names:
        if name not in catalog_by_name:
            continue
        _, comment, cols, _ = catalog_by_name[name]
        out.append(f"-- {name}: {comment}\nCREATE TABLE {name} ({', '.join(cols)});")
    text = "\n".join(out)
    tokens = len(text) // 4
    return text, tokens


section("ВАРИАНТ A: подать ВСЮ схему (60 таблиц)")
_, tok_all = schema_to_prompt([t[0] for t in SCHEMA_CATALOG])
print(f"  Токенов в промпте: ~{tok_all}")

section("ВАРИАНТ B: top-15 + FK-замыкание")
top15 = [n for n, _ in top]
final_set = sorted(fk_closure(top15))
_, tok_b = schema_to_prompt(final_set)
print(f"  Таблиц в промпте:  {len(final_set)}")
print(f"  Токенов в промпте: ~{tok_b}")
print(f"  Экономия:          {(1 - tok_b/tok_all) * 100:.0f}%")


## 6. A/B-«симуляция» — что без schema linking сломалось бы

Реально без schema linking генератор галлюцинировал бы имена
таблиц. Мы это симулируем: «выдумываем» имена, которых нет.


In [ ]:
##
# @brief Mock-генератор: на 60 таблицах без schema linking — галлюцинирует.
def mock_generate_WITHOUT_linking(question, all_tables):
    # Симулируем: при бесконечном выборе модель путает имена
    return "SELECT * FROM client_overdue_credits  -- ⚠️ такой таблицы нет!"


##
# @brief Mock-генератор: со schema linking — использует только данные.
def mock_generate_WITH_linking(question, selected_tables):
    if "clients" in selected_tables and "credit_contract" in selected_tables:
        return """SELECT c.full_name, cc.amount
FROM clients c
JOIN credit_contract cc ON cc.client_id = c.client_id
WHERE cc.status_id IN (SELECT id FROM credit_status WHERE name='overdue')"""
    return "SELECT * FROM clients  -- fallback"


section("БЕЗ schema linking — генератор галлюцинирует")
sql_bad = mock_generate_WITHOUT_linking("просроченные кредиты", [t[0] for t in SCHEMA_CATALOG])
print(f"  Сгенерировано: {sql_bad}")

section("СО schema linking — генератор знает, какие таблицы есть")
sql_good = mock_generate_WITH_linking("просроченные кредиты", final_set)
print(f"  Сгенерировано:")
for line in sql_good.split("\n"):
    print(f"    {line}")


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/01-large-schema-linking/README.md](../problems/engineering/01-large-schema-linking/README.md)
- **Варианты решения + почему так:** [problems/engineering/01-large-schema-linking/solutions.md](../problems/engineering/01-large-schema-linking/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
